# Module 8: Phương pháp điểm trong (hàm chắn) — code

Notebook đi kèm site *Tối ưu hóa phi tuyến & lồi*. Chạy tuần tự từng ô (Colab có sẵn numpy, scipy, sympy, cvxpy). Tự giải tay trước rồi đối chiếu kết quả.

## Đường trung tâm: công thức đóng và Newton (dạng C)

min x1²+x2² với x1+x2 ≥ 1: công thức t = (1+√(1+4μ))/4 và Newton có quay lui phải trùng nhau.

In [1]:
import numpy as np
def central(mu): return (1 + np.sqrt(1 + 4*mu)) / 4
def newton(mu, x):
    for _ in range(60):
        s = x.sum() - 1; g = 2*x - mu/s; H = 2*np.eye(2) + mu/s**2*np.ones((2, 2))
        if np.linalg.norm(g) < 1e-12: break
        d = np.linalg.solve(H, -g); t = 1.0
        while (x + t*d).sum() <= 1: t /= 2
        x = x + t*d
    return x
for mu in (1, .5, .1, .01, .001):
    x = newton(mu, np.array([1., 1.])); print(mu, x.round(5), round(central(mu), 5), "lambda =", round(mu/(x.sum()-1), 4))

1 [0.80902 0.80902] 0.80902 lambda = 1.618
0.5 [0.68301 0.68301] 0.68301 lambda = 1.366
0.1 [0.5458 0.5458] 0.5458 lambda = 1.0916
0.01 [0.50495 0.50495] 0.50495 lambda = 1.0099
0.001 [0.5005 0.5005] 0.5005 lambda = 1.001


## Thuật toán điểm trong đầy đủ và ảnh hưởng của β (dạng D)

Xuất phát (1,1), μ1 = 1, dừng khi m·μ < 0.005: đếm vòng ngoài và tổng bước Newton.

In [2]:
import numpy as np
def newton(mu, x):
    n = 0
    for _ in range(80):
        s = x.sum() - 1; g = 2*x - mu/s; H = 2*np.eye(2) + mu/s**2*np.ones((2, 2))
        if np.linalg.norm(g) < 1e-11: break
        d = np.linalg.solve(H, -g); t = 1.0
        while (x + t*d).sum() <= 1: t /= 2
        x = x + t*d; n += 1
    return x, n
for beta in (.5, .2, .1, .01):
    x = np.array([1., 1.]); mu = 1.0; outer = 0; tot = 0
    while True:
        x, n = newton(mu, x); outer += 1; tot += n
        if mu < 5e-3: break
        mu *= beta
    print("beta", beta, "vong ngoai", outer, "tong Newton", tot, x.round(5))

beta 0.5 vong ngoai 9 tong Newton 70 [0.50195 0.50195]
beta 0.2 vong ngoai 5 tong Newton 32 [0.5008 0.5008]
beta 0.1 vong ngoai 4 tong Newton 28 [0.5005 0.5005]
beta 0.01 vong ngoai 3 tong Newton 21 [0.50005 0.50005]


## Bài tập tr.36 và cận m·μ (dạng B, E)

min (x-2)² với 0<x<1, hàm chắn log: nghiệm bằng chia đôi; kiểm 1-x ≈ μ/2.

In [3]:
from scipy.optimize import brentq
for mu in (1, .1, .01, .001):
    r = brentq(lambda x: 2*(x-2) - mu/x + mu/(1-x), 1e-12, 1-1e-12)
    print(mu, round(r, 6), "1-x =", round(1-r, 6), "(1-x)/mu =", round((1-r)/mu, 3))

1 0.741348 1-x = 0.258652 (1-x)/mu = 0.259
0.1 0.95446 1-x = 0.04554 (1-x)/mu = 0.455
0.01 0.995049 1-x = 0.004951 (1-x)/mu = 0.495
0.001 0.9995 1-x = 0.0005 (1-x)/mu = 0.5


## Chắn nghịch đảo hay log? (min x, x ≥ 2)

Khoảng cách tới biên: sqrt(μ) so với μ.

In [4]:
import numpy as np
for mu in (1, .25, .01, 1e-4):
    print(mu, "nghịch đảo:", 2 + np.sqrt(mu), " log:", 2 + mu)

1 nghịch đảo: 3.0  log: 3
0.25 nghịch đảo: 2.5  log: 2.25
0.01 nghịch đảo: 2.1  log: 2.01
0.0001 nghịch đảo: 2.01  log: 2.0001
